# 4.1 - Deployment Model Gen AI ke Databricks Model Serving

Notebook ini mempelajari cara mendeploy model agen kecerdasan buatan (**AI Agent**) yang terdaftar di Unity Catalog ke dalam **Databricks Model Serving**, serta memanfaatkan endpoint tersebut untuk inferensi langsung dari kueri SQL analitik menggunakan fungsi `ai_query()`.

### Ringkasan Proses:
1. **Setup Lingkungan Lab**: Menginisialisasi katalog, skema, tabel sampel tiket pelanggan (`customer_support_tickets`), serta konfigurasi dari `../Includes/Lab_Setup_04`.
2. **Konfigurasi & Deployment Model Serving Endpoint**:
   - Menentukan konfigurasi inferensi: nama model di Unity Catalog, versi model (versi 2), ukuran komputasi (*workload size*), dan mengaktifkan fitur *scale-to-zero* untuk efisiensi biaya.
   - Menggunakan **Databricks SDK (`WorkspaceClient`)** untuk membuat endpoint serving terkelola dan menunggu hingga berstatus `READY`.
3. **Eksplorasi Data Tiket Dukungan Pelanggan**: Membaca dan memeriksa sampel tiket pelanggan berstatus terbuka (*open*) pada tabel Delta Lake.
4. **Inferensi Batch Menggunakan SQL AI Functions (`ai_query`)**: Menjalankan kueri SQL untuk memanggil endpoint model secara langsung pada data tabel analitik guna menjawab tiket dukungan secara otomatis.


### Langkah 1: Setup Lingkungan Lab
Menjalankan notebook setup lab untuk menginisialisasi katalog, skema, tabel `customer_support_tickets`, dan variabel pendukung sesi deployment.


In [0]:
%run ../Includes/Lab_Setup_04

### Langkah 2: Konfigurasi & Deployment Model Serving Endpoint Menggunakan Databricks SDK
Membuat endpoint Model Serving terkelola bernama `dphone_endpoint` yang melayani model `dphone_agent` (versi 2).
- `scale_to_zero_enabled = True`: mengizinkan cluster endpoint otomatis nonaktif (0 worker) saat tidak ada permintaan aktif untuk menghemat biaya komputasi.
- `workload_size = "Small"`: menentukan ukuran kapasitas komputasi cluster inferensi.
- `create_and_wait()`: membuat endpoint dan menunggu hingga statusnya menjadi `READY` dan siap melayani permintaan API.


In [0]:
from databricks.sdk.service.serving import EndpointCoreConfigInput
from databricks.sdk import WorkspaceClient

# Menentukan nama endpoint serving yang akan dibuat
serving_endpoint_name = f"dphone_endpoint"

# Menentukan nama lengkap model di Unity Catalog (catalog.schema.model) dan versinya
model_fullname = f"{course.catalog}.{course.schema}.dphone_agent"
latest_model_version = 2

# Menyusun kamus konfigurasi endpoint serving
endpoint_config_dict = {
    "served_models": [
        {
            "model_name": model_fullname,
            "model_version": latest_model_version,
            "scale_to_zero_enabled": True,  # Mengaktifkan scale-to-zero untuk efisiensi biaya komputasi
            "workload_size": "Small",        # Ukuran kapasitas komputasi cluster inferensi
            "environment_vars": {},
        },
    ]
}

# Mengonversi kamus konfigurasi ke objek EndpointCoreConfigInput Databricks SDK
endpoint_config = EndpointCoreConfigInput.from_dict(endpoint_config_dict)

# Inisialisasi WorkspaceClient dan membuat endpoint serving hingga berstatus READY
w = WorkspaceClient()
w.serving_endpoints.create_and_wait(name=serving_endpoint_name, config=endpoint_config)

### Langkah 3: Membaca Sampel Data Tiket Dukungan Pelanggan
Membaca tabel Delta `customer_support_tickets` untuk melihat contoh tiket keluhan pelanggan yang berstatus masih terbuka (*open*).


In [0]:
# Menentukan nama lengkap tabel tiket dukungan pelanggan di Unity Catalog
customer_requests_table = f"{course.catalog}.{course.schema}.customer_support_tickets"

# Membaca data tiket yang masih terbuka (status = 'open') dan mengambil 2 sampel data
customer_requests_df = spark.read.table(customer_requests_table).filter("status = 'open'").limit(2)

# Menampilkan isi DataFrame sampel tiket
display(customer_requests_df)

### Langkah 4: Inferensi Batch Menggunakan SQL AI Functions (`ai_query`)
Menggunakan fungsi bawaan Databricks SQL `ai_query()` untuk meneruskan teks keluhan pelanggan (`description`) langsung ke endpoint Model Serving `dphone_endpoint`.
Fitur ini memungkinkan pemrosesan batch dan otomatisasi tugas-tugas Gen AI langsung di lakehouse tanpa perlu menulis kode inferensi eksternal.


In [0]:
%sql
-- Menjalankan inferensi model langsung dari SQL menggunakan fungsi ai_query
-- Setiap deskripsi tiket pelanggan yang berstatus 'open' dikirim ke 'dphone_endpoint' untuk dijawab secara otomatis
SELECT description,
       ai_query(endpoint =>"dphone_endpoint", request => description) response
FROM customer_support_tickets
WHERE status = 'open'